<a href="https://colab.research.google.com/github/velascocafe23/nlp-maestria-upb/blob/main/semana-1/lab1-embeddings/lab1_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Semana 1 · Lab 1 (preprocesamiento, N-grams y embeddings)** — Sebastián Velasco Ardila · Maestría en Ciencia de Datos, UPB · Curso de NLP

Notebook guiado, ejecutado completo. Ajustes respecto al original: se eliminó un bloque duplicado de los experimentos 3.1 a 3.4 (quedó la versión corregida) y se arregló el import del tokenizador BPE (`AutoTokenizer` vive en `transformers`, no en `tokenizers`), con lo que el Tipo 3 de tokenización sí corre. Las reflexiones están en `ENTREGA.md`.

# Lab 1: Preprocesamiento, N-grams y Word Embeddings
**Curso de Maestría en NLP — Semana 1 — Viernes 9:10-10:00 PM**

| | |
|---|---|
| **Sesión** | Semana 1 · Viernes (sesiones 1.V1 y 1.V2) |
| **Hora de inicio** | 9:10 PM |
| **Duración estimada** | 50 minutos |
| **Entorno** | Google Colab gratuito — **sin GPU**, corre en CPU |

### Instrucciones para el instructor
1. Compartir el link de Colab antes de abrir la sesión
2. **Ejecutar las celdas juntos** — no dejar que avancen solos hasta la pausa
3. Pausar en cada `# === PAUSA: DISCUSIÓN ===` para discutir en grupo
4. Guardar una copia: **Archivo → Guardar una copia en Drive**

### Qué vas a aprender
Este notebook cubre **toda la jornada del viernes** en el orden en que se presentaron los conceptos:
- **Bloque 1** (diapositivas 7-8): pipeline de preprocesamiento y los 3 tipos de tokenización
- **Bloque 2** (diapositiva 10): modelos de lenguaje N-gram desde cero en Python
- **Bloque 3** (diapositivas 11-13): word embeddings, analogías y auditoría de sesgos


## Configuración del entorno

In [ ]:
# @title Configuracion — instalar librerias
!pip install -q transformers gensim scikit-learn matplotlib seaborn tokenizers nltk spacy
!python -m spacy download es_core_news_sm --quiet

import re, math, warnings, unicodedata
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
import gensim.downloader as api

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

warnings.filterwarnings('ignore')

try:
    import torch
    print(f'GPU disponible: {torch.cuda.is_available()}')
except ImportError:
    pass
print('Entorno listo.')

---
## Bloque 1 — Preprocesamiento y Tokenización
### (Diapositivas 7 y 8)

Antes de que cualquier modelo de Machine Learning vea el texto, el texto debe transformarse
a una forma que la computadora pueda procesar. Ese proceso se llama **pipeline de preprocesamiento**.

> **Regla de oro:** ningún modelo recibe texto crudo directamente.


## Experimento 1: Pipeline Completo de Limpieza de Texto
### (Diapositiva 7)

#### Que es el preprocesamiento?
El preprocesamiento transforma texto crudo en una forma normalizada que los algoritmos
pueden procesar de forma consistente. No existe un pipeline universal: las decisiones
dependen de la tarea, el idioma y el dominio.

> **Regla de oro:** ninguna tecnica de preprocesamiento es siempre buena o siempre mala.
> Cada decision tiene un trade-off entre simplificacion y perdida de informacion.

#### Tecnicas clasicas de limpieza (de mas simple a mas sofisticada)

| Tecnica | Que hace | Cuando usarla | Cuando evitarla |
|---|---|---|---|
| **Lowercase** | Iguala mayusculas | Clasificacion, BoW, embeddings | NER ("Colombia" != "colombia") |
| **Quitar acentos** | Normaliza tildes | Corpus multilingue inconsistente | NLP espanol (la tilde puede cambiar el significado) |
| **Eliminar puntuacion** | Quita simbolos | BoW, modelos de bolsa | Analisis de sentimiento ("!" puede ser clave) |
| **Normalizar numeros** | TOKEN_NUM generico | IR, clasificacion tematica | Cuando el valor exacto importa (financiero, medico) |
| **Eliminar stopwords** | Quita palabras de alta frecuencia | IR, BoW, TF-IDF | Modelos de lenguaje, NER, analisis de sentimiento |
| **Stemming** | Reduce a raiz morfologica | IR rapida, baja latencia | Cuando la forma exacta importa |
| **Lematizacion** | Forma canonicadel diccionario | Clasificacion, analisis semantico | Velocidad critica (mas lento que stemming) |

---

### Stopwords: eliminando el ruido de alta frecuencia

Las **stopwords** (palabras vacias) son terminos tan frecuentes que aportan poca
informacion discriminativa: articulos, preposiciones, conjunciones.

Ejemplo en espanol: `"el", "la", "de", "en", "y", "a", "los", "del", "se", "un"`

**Por que eliminarlas en BoW/TF-IDF:** ocupan las dimensiones con mayor frecuencia
pero no distinguen documentos. El IDF de "el" es practicamente 0 porque aparece en todos.

**Cuando NO eliminarlas:** en modelos de lenguaje y Transformers (BERT, GPT), las stopwords
son cruciales para entender la estructura gramatical. `"el banco"` y `"del banco"`
tienen significados sintacticamente distintos.

---

### Stemming: reduccion por reglas morfologicas

El **stemming** reduce cada palabra a su raiz aplicando reglas heuristicas de sufijacion.
No consulta un diccionario: aplica patrones mecanicamente.

```
"corriendo"  ->  "corr"
"corredores" ->  "corr"
"corria"     ->  "corr"     <- mismo stem: el modelo las trata como la misma palabra
```

**Ventaja:** muy rapido, sin diccionario, sin ambiguedad.
**Desventaja:** produce raices que no son palabras reales (`"corr"`),
lo que dificulta la interpretabilidad. Puede sobre-reducir:
`"banco"` y `"banquero"` pueden quedar en el mismo stem aunque tengan distintos significados.

Algoritmo clasico para espanol: **Snowball Stemmer** (NLTK).

---

### Lematizacion: forma canonica del diccionario

La **lematizacion** reduce cada palabra a su **lema**: la forma base que encontrarias
en un diccionario, usando morfologia y en algunos casos POS tagging para desambiguar.

```
"corriendo"  ->  "correr"    <- verbo en infinitivo (lema verbal)
"corredores" ->  "corredor"  <- sustantivo en singular (lema sustantivo)
"fue"        ->  "ir"        <- forma irregular del verbo ir
"bancos"     ->  "banco"     <- plural -> singular
```

**Ventaja sobre stemming:** produce palabras reales y es mas precisa linguisticamente.
**Desventaja:** mas lenta (requiere POS tagging + diccionario), dependiente del idioma.

Herramienta principal para espanol: **spaCy** con el modelo `es_core_news_sm`.

#### Diferencia clave: Stemming vs. Lematizacion

| | Stemming | Lematizacion |
|---|---|---|
| Metodo | Reglas heuristicas de sufijacion | Diccionario + POS tagging |
| Velocidad | Muy rapida (microsegundos) | Mas lenta (milisegundos) |
| Resultado | Raiz (puede no ser palabra real) | Lema (siempre palabra real) |
| Precision | Menor | Mayor |
| Idioma | Funciona bien con Snowball | spaCy tiene modelos por idioma |

**Regla practica:** usa lematizacion cuando necesitas precision y los documentos
son cortos (tweets, resenias). Usa stemming cuando tienes millones de documentos
y la latencia es critica (motores de busqueda).


In [ ]:
# @title Experimento 1: Pipeline completo de limpieza de texto
import re
import unicodedata

# Instalar lo necesario (solo la primera vez)
import subprocess
import sys

# NLTK para stopwords y stemming
try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem.snowball import SnowballStemmer
    nltk.download('stopwords', quiet=True)
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nltk'])
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem.snowball import SnowballStemmer
    nltk.download('stopwords', quiet=True)

# spaCy para lematizacion
try:
    import spacy
    try:
        nlp_es = spacy.load('es_core_news_sm')
    except OSError:
        subprocess.run([sys.executable, '-m', 'spacy', 'download', 'es_core_news_sm'], capture_output=True)
        nlp_es = spacy.load('es_core_news_sm')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'spacy'])
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'es_core_news_sm'], capture_output=True)
    import spacy
    nlp_es = spacy.load('es_core_news_sm')

print('Librerias cargadas.')

# ============================================================
# TEXTO DE EJEMPLO (el mismo de las diapositivas)
# ============================================================
texto_original = (
    'El banco BBVA Colombia S.A. reporto utilidades de $1.2 billones '
    'en el 2 trimestre de 2025, segun comunicado oficial. Excelente resultado!'
)

print('TEXTO ORIGINAL:')
print(f'  {repr(texto_original)}')
print(f'  Tokens: {len(texto_original.split())}')
print()

# ── Paso 1: Lowercase ────────────────────────────────────────
p1 = texto_original.lower()
print(f'Paso 1 — Lowercase:')
print(f'  {p1}')

# ── Paso 2: Eliminar acentos ─────────────────────────────────
def quitar_acentos(txt):
    return ''.join(c for c in unicodedata.normalize('NFD', txt)
                   if unicodedata.category(c) != 'Mn')
p2 = quitar_acentos(p1)
print()
print(f'Paso 2 — Sin acentos:')
print(f'  {p2}')

# ── Paso 3: Eliminar puntuacion y caracteres especiales ──────
p3 = re.sub(r'[^a-z0-9\s]', ' ', p2)
p3 = re.sub(r'\s+', ' ', p3).strip()
print()
print(f'Paso 3 — Sin puntuacion/especiales:')
print(f'  {p3}')

# ── Paso 4: Normalizar numeros ───────────────────────────────
p4 = re.sub(r'\b\d+\b', 'TOKEN_NUM', p3)
print()
print(f'Paso 4 — Numeros como TOKEN_NUM:')
print(f'  {p4}')

# ── Paso 5: Eliminar stopwords ───────────────────────────────
sw_es = set(stopwords.words('spanish'))
tokens_p4 = p4.split()
tokens_p5 = [t for t in tokens_p4 if t not in sw_es]
p5 = ' '.join(tokens_p5)
removidas = [t for t in tokens_p4 if t in sw_es]
print()
print(f'Paso 5 — Sin stopwords:')
print(f'  {p5}')
print(f'  Removidas: {removidas}  ({len(removidas)}/{len(tokens_p4)} tokens = {len(removidas)/len(tokens_p4)*100:.0f}%)')

# ── Paso 6: Stemming con Snowball ────────────────────────────
stemmer = SnowballStemmer('spanish')
tokens_stem = [(t, stemmer.stem(t)) for t in tokens_p5]
p6 = ' '.join(stem for _, stem in tokens_stem)
print()
print(f'Paso 6 — Stemming (Snowball):')
print(f'  {p6}')
print(f'  Cambios: {[(o, s) for o, s in tokens_stem if o != s]}')

# ── Paso 7: Lematizacion con spaCy ───────────────────────────
# Usamos el texto sin stopwords pero CON acentos para mejor precision
texto_para_spacy = ' '.join(t for t in p4.split() if t not in sw_es and t != 'token_num')
doc = nlp_es(texto_para_spacy)
tokens_lema = [(token.text, token.lemma_, token.pos_) for token in doc]
p7 = ' '.join(token.lemma_ for token in doc)
print()
print(f'Paso 7 — Lematizacion (spaCy):')
print(f'  {p7}')
print(f'  Detalle:')
for texto, lema, pos in tokens_lema:
    cambio = f' -> "{lema}"' if texto != lema else '  (sin cambio)'
    print(f'    {texto:<20} [{pos:<6}]{cambio}')

# ── Resumen comparativo ──────────────────────────────────────
print()
print('=' * 65)
print('RESUMEN DEL PIPELINE COMPLETO')
print('=' * 65)
pasos = [
    ('Original',         texto_original),
    ('1. Lowercase',     p1),
    ('2. Sin acentos',   p2),
    ('3. Sin especiales',p3),
    ('4. TOKEN_NUM',     p4),
    ('5. Sin stopwords', p5),
    ('6. Stemming',      p6),
    ('7. Lematizacion',  p7),
]
for nombre, texto in pasos:
    n = len(texto.split())
    print(f'  {nombre:<22}: {n:>2} tokens   {texto[:55]}')


### Reflexión 1

> **Pregunta:** ¿En qué escenarios NO querrías quitar los acentos del texto?
> Pista: piensa en sistemas de Named Entity Recognition (NER) o análisis de sentimiento
> en español latinoamericano vs. español peninsular.

# === PAUSA: DISCUSIÓN ===


## Experimento 2: Los 3 Tipos de Tokenización
### (Diapositiva 8)

#### ¿Qué es la tokenización?
La **tokenización** es el proceso de dividir texto en unidades mínimas llamadas **tokens**.
Es la primera decisión de diseño en cualquier sistema NLP y una de las más importantes.
El modelo nunca ve el texto completo: solo ve la secuencia de tokens representados como números.

#### ¿Por qué importa tanto?
El tamaño del vocabulario, la capacidad de manejar palabras nuevas (OOV) y la longitud
de las secuencias dependen directamente del método de tokenización.
Además, los LLMs modernos **cobran por tokens** y tienen un límite de contexto medido en tokens.

---

### Tipo 1: Por palabras
Divide el texto por espacios y puntuación. Cada palabra distinta es un token único.

**Ventajas:** Simple, intuitivo, preserva el significado de palabras completas.

**Desventajas:**
- Vocabulario enorme (100,000+ tokens para corpus grandes).
- **Problema OOV (Out-Of-Vocabulary):** si el modelo nunca vio "reportaron" durante el
  entrenamiento pero sí vio "reportó", los trata como palabras completamente distintas.
- "banco", "bancos" y "bancario" son tres tokens distintos aunque comparten raíz morfológica.

---

### Tipo 2: Por caracteres
Cada carácter individual es un token. El vocabulario es de ~100 caracteres.

**Ventajas:** Vocabulario mínimo, sin problema OOV. Puede manejar cualquier palabra.

**Desventajas:**
- Secuencias muy largas (un texto de 100 palabras puede tener 500+ tokens).
- El modelo tiene que aprender a componer significado desde letras sueltas,
  lo que requiere muchos más datos y capacidad computacional.

---

### Tipo 3: Subpalabras — BPE / WordPiece
Divide las palabras en unidades morfológicas frecuentes.
Es el método que usan **BERT, GPT, LLaMA y todos los LLMs modernos**.

El algoritmo **BPE (Byte-Pair Encoding)** parte de caracteres individuales y fusiona
iterativamente los pares más frecuentes hasta alcanzar un vocabulario objetivo:

```
"reportaron"  →  ["report", "##aron"]   (## indica sufijo)
"Colombia"    →  ["Col", "##ombia"]
"S.A."        →  ["S", ".", "A", "."]
```

**Ventajas:**
- Balance óptimo: vocabulario manejable (~30,000-50,000) sin problema OOV real.
- Las palabras frecuentes son 1 solo token; las raras se descomponen en partes conocidas.
- Captura morfología: "correr", "corriendo", "corrió" comparten el prefijo "corr".

**Desventajas:**
- "1 token" no equivale a "1 palabra" — hay que tenerlo en cuenta en sistemas de búsqueda.
- Textos en español suelen generar más tokens que en inglés para el mismo contenido,
  porque los tokenizadores se optimizan mayoritariamente para inglés.

#### Implicación práctica para LLMs
Un documento de 10 páginas en español puede tener 5,000-8,000 tokens.
Si el modelo tiene un límite de 8,096 tokens, el documento llena casi todo el contexto.
Esto importa al diseñar sistemas RAG, que veremos en la Semana 3.


In [ ]:
# @title Experimento 2: Comparativa de los 3 tipos de tokenización

texto = 'El banco BBVA Colombia reportó utilidades en el trimestre'

# Tipo 1: Por palabras
tokens_palabras = texto.lower().split()
print('TIPO 1 — Por palabras:')
print(f'  N° tokens: {len(tokens_palabras)}')
print(f'  Tokens:    {tokens_palabras}')

# Tipo 2: Por caracteres
tokens_chars = list(texto.lower().replace(' ', '_'))
print()
print('TIPO 2 — Por caracteres:')
print(f'  N° tokens: {len(tokens_chars)}')
print(f'  Primeros 25: {tokens_chars[:25]} ...')

# Tipo 3: BPE con BERT multilingüe
print()
print('TIPO 3 — BPE/WordPiece con bert-base-multilingual-cased...')
try:
    from transformers import AutoTokenizer  # (el original importaba de `tokenizers`, que no tiene AutoTokenizer)
    tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
    tokens_bpe = tokenizer.tokenize(texto)
    print(f'  N° tokens: {len(tokens_bpe)}')
    print(f'  Tokens:    {tokens_bpe}')
    print()
    print('Cómo BPE descompone palabras poco frecuentes:')
    for word in ['reportaron', 'descontextualizado', 'bancolombia', 'fintech']:
        t = tokenizer.tokenize(word)
        print(f'  {word:<30} → {t}')
except Exception as e:
    print(f'  Error al cargar tokenizador: {e}')
    tokens_bpe_n = 8  # aproximado
    print(f'  (Ejemplo ilustrativo: ~{tokens_bpe_n} tokens para este texto)')

# Tabla comparativa
print()
print('=' * 65)
print(f'{"Método":<22} {"N° tokens":>10}  {"Uso en LLMs":<30}')
print('=' * 65)
print(f'{"Por palabras":<22} {len(tokens_palabras):>10}  {"Raramente (OOV problem)":<30}')
print(f'{"Por caracteres":<22} {len(tokens_chars):>10}  {"Casos especiales (ADN, código)":<30}')
print(f'{"BPE/WordPiece":<22} {len(tokens_bpe) if "tokens_bpe" in dir() else "~8":>10}  {"BERT, GPT, LLaMA, T5":<30}')

### Reflexión 2

> **Pregunta:** Tienes un sistema de búsqueda en documentos legales colombianos.
> Un abogado escribe "desistimiento" pero el modelo fue entrenado con pocos documentos
> legales y no tiene ese token en el vocabulario.
>
> 1. ¿Qué pasa si usas tokenización por palabras?
> 2. ¿Qué pasa si usas BPE?
> 3. ¿En qué caso te conviene tokenizar por caracteres para este dominio?

# === PAUSA: DISCUSIÓN ===


---
## Bloque 1.5 — Representaciones Clasicas de Texto
### (NLP Clasico: One-Hot, Bag of Words, TF-IDF, OOV)

Antes de los embeddings, los sistemas NLP representaban texto de forma mucho mas simple.
Entender estas representaciones es esencial porque:
1. Son la base de muchos clasificadores en produccion hoy
2. Permiten entender exactamente **que informacion se pierde** en cada representacion
3. Explican por que los embeddings fueron un avance tan grande


---
## Bloque 1.5 — Representaciones Clásicas de Texto
### (NLP Clásico: One-Hot, Bag of Words, TF-IDF, OOV)

Antes de los embeddings, los sistemas NLP representaban texto de forma mucho más simple.
Entender estas representaciones clásicas es esencial porque:
1. Son la base de muchos clasificadores en producción hoy
2. Permiten entender exactamente **qué información se pierde** en cada representación
3. Explican por qué los embeddings fueron un avance tan grande


## Experimento 3.1: One-Hot Encoding

#### ¿Qué es el One-Hot Encoding?
El **one-hot encoding** es la representación más simple de una palabra como número:
un vector de longitud igual al tamaño del vocabulario, con un 1 en la posición
correspondiente a esa palabra y 0 en todas las demás.

```
Vocabulario: ["el", "banco", "sube", "tasa"]

"el"    → [1, 0, 0, 0]
"banco" → [0, 1, 0, 0]
"sube"  → [0, 0, 1, 0]
"tasa"  → [0, 0, 0, 1]
```

#### ¿Por qué fue el primer intento?
Es la forma más directa de convertir un símbolo discreto (una palabra) en un número.
Cualquier algoritmo de machine learning puede trabajar con vectores numéricos,
así que one-hot permite usar regresión logística, SVM, redes neuronales, etc. sobre texto.

#### Intuición: interruptores encendido/apagado
Imagina que tienes un panel con un interruptor por cada palabra del diccionario.
Cuando ves una palabra, enciendes su interruptor y apagas todos los demás.
El estado del panel es el one-hot vector de esa palabra.

#### Limitaciones críticas
- **Alta dimensionalidad:** si el vocabulario tiene 50,000 palabras, cada vector tiene 50,000 dimensiones
  con 49,999 ceros. Esto se llama un vector **disperso (sparse)**.
- **No captura similitud:** `similitud("banco", "entidad") = 0` igual que `similitud("banco", "zapato") = 0`.
  Todas las palabras distintas son equidistantes.
- **No escala:** con un corpus de millones de palabras únicas, los vectores son computacionalmente prohibitivos.
- **OOV total:** si una palabra no estaba en el vocabulario de entrenamiento,
  no tiene representación posible.


In [ ]:
# @title Experimento 3.1: One-Hot Encoding en Python

# Vocabulario de ejemplo
vocabulario = ['el', 'banco', 'sube', 'la', 'tasa', 'baja', 'central', 'reporto']
V = len(vocabulario)
tok2idx = {tok: i for i, tok in enumerate(vocabulario)}

def one_hot(palabra, vocabulario, tok2idx):
    vector = [0] * len(vocabulario)
    if palabra in tok2idx:
        vector[tok2idx[palabra]] = 1
    return vector

# Mostrar vectores
print('ONE-HOT ENCODING')
print(f'Vocabulario: {vocabulario}')
print(f'Dimensión de cada vector: {V}')
print()
print(f'{"Palabra":<12} Vector')
print('-' * 55)
for palabra in ['banco', 'tasa', 'central']:
    v = one_hot(palabra, vocabulario, tok2idx)
    print(f'{palabra:<12} {v}')

# Demostrar que la similitud entre cualquier par de palabras distintas es siempre 0
import numpy as np
pares = [('banco', 'tasa'), ('banco', 'central'), ('banco', 'sube')]
print()
print('Similitud (producto punto) entre pares:')
for w1, w2 in pares:
    v1 = np.array(one_hot(w1, vocabulario, tok2idx))
    v2 = np.array(one_hot(w2, vocabulario, tok2idx))
    sim = np.dot(v1, v2)
    print(f'  sim("{w1}", "{w2}") = {sim}')

print()
print('Conclusión: con one-hot, todas las palabras distintas son equidistantes.')
print('No hay forma de saber que "banco" y "tasa" están más relacionadas que "banco" y "sube".')


## Experimento 3.2: Bag of Words (BoW) y TF-IDF

#### ¿Qué es Bag of Words?
**Bag of Words (BoW)** o "bolsa de palabras" representa un **documento** (no una palabra)
como un vector de frecuencias: cuántas veces aparece cada palabra del vocabulario en ese documento.
El orden de las palabras se ignora completamente.

```
Documento A: "el banco sube el banco"
BoW(A):      {"el": 2, "banco": 2, "sube": 1, ...}
Vector(A):   [2, 2, 1, 0, 0, 0, 0, 0]  (en el orden del vocabulario)

Documento B: "sube el banco el banco"
BoW(B):      {"el": 2, "banco": 2, "sube": 1, ...}
Vector(B):   [2, 2, 1, 0, 0, 0, 0, 0]  ← MISMO VECTOR que A!
```

#### ¿Por qué importa?
A pesar de su simplicidad, BoW funciona sorprendentemente bien para:
- Clasificación de texto (spam, sentimiento, categorías)
- Recuperación de información (búsqueda de documentos)
- Clustering de documentos

Es la base de algoritmos clásicos como Naive Bayes y SVM para texto, que siguen siendo
competitivos en muchos benchmarks cuando los datos son escasos.

#### TF-IDF: BoW con peso por importancia
**TF-IDF (Term Frequency - Inverse Document Frequency)** mejora BoW ponderando cada término:
palabras que aparecen en todos los documentos ("el", "de", "la") reciben bajo peso;
palabras discriminativas reciben alto peso.

```
TF(t, d)  = frecuencia del término t en el documento d
IDF(t)    = log(N / df(t))   donde N = total documentos, df = docs que contienen t
TF-IDF(t, d) = TF(t, d) × IDF(t)
```

- "banco" aparece en 2 de 10 docs → IDF alto → peso alto
- "el" aparece en los 10 docs → IDF = log(10/10) = 0 → peso cero

#### Limitaciones de BoW y TF-IDF
- **Pierde el orden:** "no me gustó" y "me gustó no" tienen el mismo vector BoW.
  Esto es un problema grave para análisis de sentimiento con negaciones.
- **Alta dimensionalidad:** el vector tiene tantas dimensiones como palabras en el vocabulario.
- **Sin semántica:** "banco" y "entidad financiera" siguen siendo distintos a pesar de significar lo mismo.
- **OOV:** palabras nuevas no tienen representación.


In [ ]:
# @title Experimento 3.2: BoW y TF-IDF con sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np

documentos = [
    'el banco central sube la tasa de interes',
    'el banco reporto utilidades record',
    'la tasa de cambio afecto la economia',
    'el banco anuncio nuevas medidas',
    'la economia colombiana crecio',
]

# ── BoW con CountVectorizer ─────────────────────────────────
cv = CountVectorizer()
X_bow = cv.fit_transform(documentos)
print('VOCABULARIO BoW:')
print(f'  {cv.get_feature_names_out().tolist()}')
print()
print('MATRIZ BoW (filas=documentos, columnas=palabras):')
print(X_bow.toarray())
print(f'Dimensión: {X_bow.shape}  ({X_bow.shape[0]} docs x {X_bow.shape[1]} términos)')

# ── TF-IDF ──────────────────────────────────────────────────
print()
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(documentos)
print('PESOS TF-IDF del documento 1 ("el banco central sube la tasa..."):')
terms = tfidf.get_feature_names_out()
scores = X_tfidf[0].toarray()[0]
nonzero = [(terms[i], round(scores[i], 3)) for i in np.argsort(scores)[::-1] if scores[i] > 0]
for term, score in nonzero:
    barra = '|' * int(score * 20)
    print(f'  {term:<15} {score:.3f}  {barra}')

print()
print('Observa: términos como "el" o "la" tienen peso bajo o cero')
print('porque aparecen en muchos documentos -> IDF bajo.')


## Experimento 3.3: BoW + TF-IDF + Machine Learning

#### El pipeline clásico de NLP para clasificación
Antes de los Transformers, el flujo estándar para clasificación de texto era:

```
Texto crudo  →  CountVectorizer/TfidfVectorizer  →  Clasificador (LogReg, SVM, NB)  →  Etiqueta
```

Este pipeline es:
- Muy rápido (millisegundos para miles de documentos)
- Interpretable (podemos ver qué palabras mueven la predicción)
- Competitivo con LLMs cuando los datos de entrenamiento son escasos (< 1,000 ejemplos)

#### Por qué estudiar esto aunque existan los LLMs
- En producción con millones de documentos/día, el costo de inferencia con LLMs es prohibitivo
- Para tareas bien definidas (clasificar correos de soporte, detectar spam), BoW+LogReg
  frecuentemente alcanza el 90-95% del rendimiento de BERT con el 1% del costo computacional
- Los coeficientes del modelo son directamente interpretables: las 10 palabras más predictivas
  para cada clase

#### El dataset
28 tweets de sentimiento en español (14 positivos, 14 negativos), incluidos directamente
en el notebook para que no dependas de ninguna fuente externa.


In [ ]:
# @title Experimento 3.3: Clasificador de sentimiento con BoW + TF-IDF + LogReg
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
import numpy as np

# Dataset inline: tweets de sentimiento en español
positivos = [
    'excelente servicio al cliente muy recomendado',
    'producto de muy buena calidad lo recomiendo',
    'estoy muy satisfecho con mi compra',
    'funciona perfectamente supero mis expectativas',
    'entrega rapida y producto en perfectas condiciones',
    'increible relacion calidad precio',
    'el mejor producto que he comprado este anio',
    'muy contento con el resultado final',
    'soporte tecnico amable y eficiente',
    'sin duda volvere a comprar aqui',
    'me encanto totalmente satisfecho',
    'superb experiencia de compra la recomiendo',
    'calidad excepcional precio justo',
    'llego rapido bien empacado excelente',
]
negativos = [
    'muy mala experiencia no lo recomiendo',
    'el producto llego danado y tardo demasiado',
    'pesima calidad una completa decepcion',
    'el servicio al cliente fue horrible',
    'no funciona como se prometia en la descripcion',
    'perdi mi dinero absolutamente inutil',
    'la peor compra que he hecho en mi vida',
    'nunca volvere a comprar en esta tienda',
    'producto defectuoso y sin garantia',
    'estafa total no sirve para nada',
    'pesimo envio llego roto y sucio',
    'atencion al cliente desastrosa no ayudan',
    'precio alto calidad terrible evitar',
    'completamente decepcionante muy malo',
]

textos = positivos + negativos
etiquetas = [1]*len(positivos) + [0]*len(negativos)  # 1=positivo, 0=negativo

# Train/test split simple (80/20)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    textos, etiquetas, test_size=0.25, random_state=42, stratify=etiquetas
)

# Baseline: predecir siempre la clase mayoritaria
baseline_pred = [max(set(y_train), key=y_train.count)] * len(y_test)
acc_baseline = accuracy_score(y_test, baseline_pred)

# Pipeline BoW (solo frecuencias)
pipe_bow = Pipeline([
    ('vec', TfidfVectorizer(use_idf=False, norm=None)),
    ('clf', LogisticRegression(random_state=42, max_iter=200))
])
pipe_bow.fit(X_train, y_train)
acc_bow = accuracy_score(y_test, pipe_bow.predict(X_test))

# Pipeline TF-IDF
pipe_tfidf = Pipeline([
    ('vec', TfidfVectorizer()),
    ('clf', LogisticRegression(random_state=42, max_iter=200))
])
pipe_tfidf.fit(X_train, y_train)
acc_tfidf = accuracy_score(y_test, pipe_tfidf.predict(X_test))

# Resultados
print('=' * 50)
print('COMPARATIVA DE MODELOS')
print('=' * 50)
print(f'Baseline (clase mayoritaria): {acc_baseline:.2f}')
print(f'BoW + LogReg:                 {acc_bow:.2f}')
print(f'TF-IDF + LogReg:              {acc_tfidf:.2f}')

print()
print('Reporte detallado (TF-IDF + LogReg):')
print(classification_report(y_test, pipe_tfidf.predict(X_test),
      target_names=['Negativo', 'Positivo']))

# Palabras más importantes por clase
print('Top 8 palabras más predictivas:')
vec = pipe_tfidf.named_steps['vec']
clf = pipe_tfidf.named_steps['clf']
nombres = vec.get_feature_names_out()
coefs = clf.coef_[0]

top_pos = np.argsort(coefs)[-8:][::-1]
top_neg = np.argsort(coefs)[:8]

print('  Positivo:', [(nombres[i], round(coefs[i], 2)) for i in top_pos])
print('  Negativo:', [(nombres[i], round(coefs[i], 2)) for i in top_neg])


### Reflexión 3.5 — BoW y sus limitaciones

> **1. Orden de palabras:** La oración "no me gustó el banco" tiene el mismo vector BoW que
> "me gustó el banco, no". ¿Qué consecuencias tiene esto para un sistema de análisis de sentimiento?
>
> **2. Interpretabilidad:** Los coeficientes del modelo de regresión logística muestran exactamente
> qué palabras predicen cada clase. ¿En qué escenarios de negocio es valiosa esta interpretabilidad?
>
> **3. Escalabilidad:** El pipeline TF-IDF + LogReg clasifica 10,000 documentos en milisegundos;
> BERT tarda varios segundos. ¿Qué criterios usarías para decidir cuándo usar cada uno?

# === PAUSA: DISCUSIÓN ===


## Experimento 3.4: OOV — Out-Of-Vocabulary

#### ¿Qué es OOV?
**OOV (Out-Of-Vocabulary)** se produce cuando un modelo encuentra durante la inferencia
una palabra que **no estaba en el vocabulario de entrenamiento** y por lo tanto no tiene
ninguna representación numérica.

Es uno de los problemas más comunes y costosos de los sistemas NLP en producción:
nombres propios nuevos, neologismos, jerga, errores tipográficos y términos técnicos
generan constantemente palabras fuera del vocabulario.

#### ¿Por qué es un problema?
- Con **One-Hot / BoW**: la palabra no existe como dimensión → el modelo no puede hacer nada con ella
- Con **Word2Vec / GloVe**: no hay vector asociado → el modelo lanza un error o ignora la palabra
- Con **N-grams**: el bigrama no fue visto → probabilidad 0 (antes de smoothing)

#### Intuición: el taxista sin mapa
Imagina un taxista que solo conoce las calles de Medellín que existían en 2020.
Si le pides que vaya a una calle nueva inaugurada en 2024, no tiene ruta.
Eso es OOV: el modelo solo conoce el "mapa" de palabras que vio durante el entrenamiento.

#### Soluciones al problema OOV

| Solución | Idea | Ventaja | Desventaja |
|---|---|---|---|
| **Token `<UNK>`** | Reemplazar OOV por un token especial | Simple | Pierde toda la información de la palabra |
| **BPE / WordPiece** | Descomponer en subpalabras conocidas | Casi elimina OOV | Altera la granularidad |
| **Embeddings de caracteres** | Aprender representaciones de caracteres | Maneja cualquier palabra | Más complejo |
| **Corpus más grande** | Entrenar con más datos | Reduce OOV natural | Costoso, no elimina OOV futuro |

#### Conexión con los experimentos anteriores
- **One-Hot / BoW** (exp. 3.1-3.3): OOV severo — vocabulario cerrado, sin recurso
- **N-grams** (exp. 3, Bloque 2): Laplace smoothing solo asigna probabilidad mínima, no representación
- **GloVe** (exp. 4, Bloque 3): OOV total para palabras no vistas
- **BPE (BERT)** (exp. 2, Bloque 1): casi sin OOV porque descompone en caracteres conocidos
  — esta es una de las razones por las que todos los LLMs modernos usan BPE


In [ ]:
# @title Experimento 3.4: Demostración de OOV en la práctica
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Vocabulario de entrenamiento: corpus de noticias financieras 2024
corpus_entrenamiento = [
    'el banco central subio la tasa de interes',
    'el banco reporto utilidades record este trimestre',
    'la economía colombiana mostro crecimiento sostenido',
    'el dolar se fortalecio frente al peso colombiano',
    'los depositos bancarios crecieron un cinco por ciento',
]

# Entrenar el vectorizador en el corpus de 2024
cv = CountVectorizer()
cv.fit(corpus_entrenamiento)
vocab = set(cv.get_feature_names_out())

print('VOCABULARIO APRENDIDO EN ENTRENAMIENTO:')
print(f'  {sorted(vocab)}')
print(f'  Total: {len(vocab)} palabras')

# Textos de prueba con palabras nuevas de 2025-2026 (no estaban en el corpus)
textos_nuevos = [
    'el banco lanzo un nuevo producto fintech',  # "fintech" es OOV
    'la criptomoneda bitcoin bajo esta semana',   # "criptomoneda" y "bitcoin" son OOV
    'nequi y daviplata lideran pagos digitales',  # "nequi", "daviplata" son OOV
    'el banco central subio la tasa',             # sin OOV — palabras conocidas
]

print()
print('ANÁLISIS DE OOV EN TEXTOS NUEVOS:')
print('=' * 60)

for texto in textos_nuevos:
    tokens = texto.lower().split()
    oov = [t for t in tokens if t not in vocab]
    conocidas = [t for t in tokens if t in vocab]
    pct_oov = len(oov)/len(tokens)*100
    print(f'Texto: "{texto}"')
    print(f'  Palabras conocidas: {conocidas}')
    print(f'  OOV ({len(oov)}/{len(tokens)}, {pct_oov:.0f}%): {oov}')
    # El vector transformado ignora las palabras OOV
    vec = cv.transform([texto]).toarray()[0]
    tokens_capturados = sum(vec)
    print(f'  Tokens capturados en el vector: {int(tokens_capturados)}')
    print()

# Solución 1: token <UNK>
print('SOLUCIÓN 1 — Token <UNK>:')
def con_unk(texto, vocab):
    return ' '.join(t if t in vocab else '<UNK>' for t in texto.lower().split())

for texto in textos_nuevos[:2]:
    print(f'  Original: "{texto}"')
    print(f'  Con UNK:  "{con_unk(texto, vocab)}"')

print()
print('Limitación de <UNK>: "fintech", "criptomoneda" y "bitcoin" se convierten en el mismo token.')
print('El modelo pierde toda distinción entre ellos.')
print()
print('Solución real: BPE (Byte-Pair Encoding) — lo que vimos en el Experimento 2.')
print('BPE descompone "fintech" → ["fin", "##tech"] — partes que probablemente sí existen.')


### Reflexión 3.6 — OOV

> **1. OOV en producción:** Tu sistema de análisis de noticias financieras fue entrenado en 2023.
> En 2025 aparecen nombres de empresas colombianas nuevas (IA startups, fintechs).
> ¿Cómo manejarías el OOV sin reentrenar desde cero?
>
> **2. BPE como solución:** "Bancolombia" no estaba en el vocabulario de GloVe
> pero sí "banco" y "colombia". Con BPE, "Bancolombia" → ["Bank", "##co", "##lombia"].
> ¿Esas subpalabras capturan el significado de la empresa completa?
>
> **3. Conexión hacia BERT:** BERT usa WordPiece (similar a BPE) y embeddings contextualizados.
> ¿Cómo crees que BERT resuelve simultáneamente OOV Y el problema de los embeddings estáticos?

# === PAUSA: DISCUSIÓN ===


---
## Bloque 2 — Modelos de Lenguaje N-gram
### (Diapositiva 10)

## Experimento 3: N-gram Language Model desde Cero

#### ¿Qué es un modelo de lenguaje?
Un **modelo de lenguaje** asigna una probabilidad a cualquier secuencia de palabras.
Nos permite responder: ¿qué tan probable es que alguien diga
"el banco reportó utilidades" frente a "el banco reportó zapatos"?

Formalmente:
```
P(w₁, w₂, ..., wₙ) = P(w₁) × P(w₂|w₁) × P(w₃|w₁,w₂) × ...
```

#### ¿Por qué importan los N-grams?
Son la base histórica de los sistemas de autocompletado, corrección ortográfica,
reconocimiento de voz y traducción automática. Aunque los Transformers los superaron,
siguen siendo útiles en:
- Sistemas con recursos computacionales limitados
- Evaluación de modelos (la métrica **perplexidad** sigue siendo estándar)
- Comprensión intuitiva de cómo funciona la probabilidad sobre secuencias

#### Intuición: el modelo como contador de estadísticas
Imagina que lees 10,000 artículos de noticias financieras en español.
Cada vez que ves "banco" seguida de "central", lo apuntas en un cuaderno.
Si "banco" apareció 5,000 veces y "banco central" 800 veces:

```
P("central" | "banco") = count("banco central") / count("banco") = 800/5000 = 0.16
```

Eso es exactamente lo que hace un modelo **bigrama** (N=2).

#### La familia N-gram
| Nombre | N | Contexto |
|--------|---|----------|
| Unigrama | 1 | Ninguno — solo frecuencia de cada palabra |
| Bigrama | 2 | La palabra anterior |
| Trigrama | 3 | Las 2 palabras anteriores |
| 4-grama | 4 | Las 3 palabras anteriores |

Mayor N = más contexto = mejor calidad, pero necesita más datos y memoria.

#### Problema de los ceros y Laplace Smoothing
Si "banco sinestésico" nunca apareció en el corpus, su probabilidad es 0.
Esto hace que toda la probabilidad de una oración sea 0 en cuanto aparece una secuencia no vista.

**Solución — Laplace smoothing (add-1):**
```
P_smooth(w | contexto) = (count(contexto, w) + 1) / (count(contexto) + |V|)
```
Donde |V| es el tamaño del vocabulario.

#### Perplexidad: cómo medir un modelo de lenguaje
La **perplexidad** mide qué tan bien el modelo predice texto no visto.
Intuitivamente: cuántas palabras igualmente probables tiene el modelo en mente en cada paso.

```
Perplexidad = exp(−(1/N) × Σ log P(wᵢ | contexto))
```

- Perplexidad baja = modelo más seguro, mejor predicción
- Un modelo aleatorio sobre vocabulario de 50,000 palabras tiene perplexidad = 50,000
- Un buen modelo de noticias financieras tiene perplexidad de ~100-500

#### Limitaciones
- Solo ven N-1 palabras de contexto: no capturan dependencias de largo alcance.
- No generalizan: "banco colombiano" y "entidad financiera colombiana" son
  secuencias completamente distintas aunque signifiquen lo mismo.
- Requieren corpus muy grandes para N > 3 (demasiados posibles N-gramas).


In [ ]:
# @title Experimento 3: N-gram Language Model en Python

# Corpus: 20 oraciones de noticias financieras en español
corpus_texto = [
    'el banco central subio la tasa de interes',
    'el banco reporto utilidades record en el trimestre',
    'la tasa de cambio del dolar afecto la economia colombiana',
    'el banco central anuncio nuevas medidas monetarias',
    'los bancos privados reportaron perdidas este semestre',
    'la economia colombiana crecio un tres por ciento',
    'el banco de la republica mantuvo la tasa de interes',
    'las utilidades del sector financiero superaron expectativas',
    'el gobierno anuncio nuevos subsidios para el sector bancario',
    'la tasa de desempleo bajo gracias al crecimiento economico',
    'el banco central intervino en el mercado cambiario',
    'los fondos de inversion reportaron rendimientos positivos',
    'la cartera de credito de los bancos crecio este año',
    'el banco emisor publico su informe de politica monetaria',
    'las tasas de interes hipotecarias bajaron este mes',
    'el sector bancario colombiano mostro solidez financiera',
    'la superintendencia financiera regulo las tasas de credito',
    'los depositos bancarios crecieron en el ultimo trimestre',
    'el banco anuncio una nueva linea de credito empresarial',
    'la cartera vencida del sistema bancario se redujo',
]

# Tokenización
def tokenizar(oracion):
    return ['<s>'] + oracion.lower().split() + ['</s>']

corpus_tokens = [tokenizar(o) for o in corpus_texto]
todas_palabras = [tok for oracion in corpus_tokens for tok in oracion]

conteo_palabras = Counter(todas_palabras)
vocab = set(conteo_palabras.keys())
V = len(vocab)
print(f'Vocabulario: {V} palabras únicas')
print(f'Total tokens: {len(todas_palabras)}')

# Conteos de bigramas
conteo_bigramas = Counter()
conteo_unigramas = Counter()
for oracion in corpus_tokens:
    for i in range(len(oracion)):
        conteo_unigramas[oracion[i]] += 1
    for i in range(len(oracion) - 1):
        conteo_bigramas[(oracion[i], oracion[i+1])] += 1

# Probabilidad bigrama con Laplace smoothing
def prob_bigrama(w2, w1, alpha=1):
    count_bg = conteo_bigramas.get((w1, w2), 0)
    count_ug = conteo_unigramas.get(w1, 0)
    return (count_bg + alpha) / (count_ug + alpha * V)

# Top-10 palabras más probables después de 'banco'
print()
print("Top 10 palabras más probables después de 'banco':")
candidatas = [(w, prob_bigrama(w, 'banco')) for w in vocab if w not in ['<s>', '</s>']]
candidatas.sort(key=lambda x: x[1], reverse=True)
for i, (palabra, prob) in enumerate(candidatas[:10], 1):
    barra = '|' * int(prob * 500)
    print(f'  {i:2}. {palabra:<20} P={prob:.4f}  {barra}')

# Perplexidad
def perplexidad(oracion_str):
    tokens = tokenizar(oracion_str)
    N = len(tokens) - 1
    log_prob = sum(math.log(prob_bigrama(tokens[i], tokens[i-1])) for i in range(1, len(tokens)))
    return math.exp(-log_prob / N)

print()
print('Perplexidad en oraciones de prueba:')
oraciones_prueba = [
    'el banco reporto utilidades en el trimestre',
    'la tasa de interes subio',
    'el perro come pizza en el parque',
]
for oracion in oraciones_prueba:
    ppl = perplexidad(oracion)
    tipo = '(coherente con corpus)' if ppl < 1000 else '(incoherente para corpus financiero)'
    print(f'  PPL={ppl:>8.1f}  {tipo:<35} "{oracion}"')

print()
print('Interpretación:')
print('  PPL baja = el modelo predice bien esta secuencia (coherente con el corpus)')
print('  PPL alta = el modelo nunca vio algo parecido (secuencia improbable)')


### Reflexión 3

> **Pregunta:** La perplexidad de "el perro come pizza en el parque" es mucho más alta
> que la de "el banco reportó utilidades". ¿Qué dice eso sobre el modelo?
>
> Bonus: ¿Por qué no podemos usar P=0 para bigramas que nunca aparecieron?
> ¿Qué pasaría matemáticamente con la perplexidad de toda la oración?

# === PAUSA: DISCUSIÓN ===


---
## Bloque 3 — Word Embeddings con GloVe
### (Diapositivas 11-13)

## Experimento 4: Cargando Embeddings GloVe

#### ¿Qué son los word embeddings?
Un **word embedding** es una representación densa de una palabra como un vector de números reales.
A diferencia de one-hot encoding (un vector con un solo 1 y miles de 0s),
los embeddings son vectores de 50-300 dimensiones donde cada dimensión captura
algún aspecto semántico o sintáctico.

#### ¿Por qué superan a los N-grams?
Los N-grams tratan cada palabra como un símbolo atómico sin relación con otros.
Los embeddings capturan **similitud semántica**: "banco" y "entidad financiera"
quedan cerca en el espacio vectorial porque aparecen en contextos similares.

#### Hipótesis distribucional (Firth, 1957)
> *"You shall know a word by the company it keeps."*
>
> Conoces una palabra por la compañía que mantiene.

Si "banco" y "entidad financiera" aparecen rodeados de las mismas palabras en millones de textos,
sus vectores convergerán hacia la misma región del espacio.

#### GloVe (Global Vectors for Word Representation)
GloVe factoriza la **matriz de co-ocurrencia global** del corpus: una matriz donde la celda (i, j)
cuenta cuántas veces la palabra i aparece en el contexto de la palabra j en todo el corpus.
Fue entrenado sobre 6 mil millones de tokens de Wikipedia y Gigaword.
Cada palabra queda representada por un vector de 100 dimensiones (en la versión que usamos).

#### GloVe vs. Word2Vec
- **Word2Vec** (Mikolov et al., 2013): red neuronal con ventana local de contexto
- **GloVe** (Pennington et al., 2014): factorización de la matriz global de co-ocurrencia

En la práctica, ambos producen embeddings de calidad similar.

#### Limitación crítica: embeddings estáticos
Ambos producen **embeddings estáticos**: "banco" tiene el mismo vector en
"fui al banco a depositar" y en "me senté en el banco del parque".
Esta es la limitación fundamental que **BERT resuelve la semana que viene**.


In [ ]:
# @title Experimento 4: Cargar GloVe (≈ 2 min de descarga)
print('Cargando embeddings GloVe de 100 dimensiones...')
print('(Entrenado en 6 mil millones de tokens de Wikipedia + Gigaword)')
modelo = api.load('glove-wiki-gigaword-100')
print(f'Modelo cargado. Vocabulario: {len(modelo.key_to_index):,} palabras')
print(f'Dimensiones: {modelo.vector_size}')
print()
print("Vector de 'bank' (primeras 10 dimensiones):")
print(f'  {modelo["bank"][:10].round(3)}')
print()
print('Nota: GloVe fue entrenado en inglés.')
print('Para español existen FastText (Meta) y BETO (Uniandes) — los veremos en Semana 2.')


## Experimento 5: Similitud Semántica

#### Similitud de coseno
La similitud entre dos vectores se mide con el **coseno del ángulo** entre ellos:

```
similitud_coseno(A, B) = (A · B) / (|A| × |B|)
```

- **Valor cercano a 1**: vectores en la misma dirección → palabras semánticamente similares
- **Valor cercano a 0**: vectores perpendiculares → sin relación semántica
- **Valor negativo**: vectores opuestos → relación antónima (poco común en GloVe)

Esta métrica es preferida sobre la distancia euclidiana porque normaliza por la magnitud,
así una palabra que aparece más frecuentemente no tiene vecinos más cercanos solo por eso.


In [ ]:
# @title Experimento 5: Similitud semántica entre palabras
print('=' * 60)
print(f'{"Par de palabras":<28} {"Similitud":>9}  Relación')
print('=' * 60)

pares = [
    ('bank', 'credit',     'financiero-financiero'),
    ('bank', 'loan',       'financiero-financiero'),
    ('bank', 'rate',       'financiero-financiero'),
    ('bank', 'investment', 'financiero-financiero'),
    ('bank', 'cat',        'financiero vs. animal'),
    ('bank', 'dog',        'financiero vs. animal'),
    ('king', 'queen',      'royalty'),
    ('man',  'woman',      'género'),
]

for w1, w2, relacion in pares:
    if w1 in modelo and w2 in modelo:
        sim = modelo.similarity(w1, w2)
        barra = '|' * int(sim * 20)
        espacio = ' ' * max(0, 20 - len(w1) - len(w2))
        print(f'  ({w1}, {w2}){espacio} {sim:>8.3f}  {barra}')

print()
print("Palabras más similares a 'bank':")
for w, s in modelo.most_similar('bank', topn=8):
    print(f'  {w:<15} {s:.3f}')


## Experimento 6: Analogías Vectoriales

#### La propiedad emergente más famosa de Word2Vec/GloVe

Si los embeddings capturan relaciones semánticas, deberían poder resolverse con aritmética vectorial:

```
rey - hombre + mujer ≈ reina
```

Esto funciona porque el modelo aprendió que rey↔reina tiene la misma relación que hombre↔mujer,
sin que nadie le dijera explícitamente que esas categorías existen.
**Emerge del entrenamiento en corpus masivos, sin supervisión alguna.**

#### Ejemplo LATAM
La misma propiedad aplica a relaciones geográficas:

```
Colombia - Bogotá + Lima ≈ Perú
```

El modelo infiere que "Lima es a Perú lo que Bogotá es a Colombia"
porque los textos sobre capitales de países siguen patrones similares en todo el corpus.


In [ ]:
# @title Experimento 6: Analogías vectoriales
def analogia(positivos, negativos, topn=5):
    try:
        return modelo.most_similar(positive=positivos, negative=negativos, topn=topn)
    except KeyError as e:
        return [(f'Palabra no encontrada: {e}', 0.0)]

analogias = [
    (['king', 'woman'],   ['man'],      'rey - hombre + mujer'),
    (['france', 'berlin'],['germany'],  'Francia - Alemania + Berlín'),
    (['paris', 'germany'],['france'],   'París - Francia + Alemania'),
    (['colombia', 'lima'],['bogota'],   'Colombia - Bogotá + Lima  (LATAM)'),
    (['bank', 'credit'],  ['money'],    'banco - dinero + crédito'),
]

print('ANALOGÍAS VECTORIALES')
print('=' * 55)
for positivos, negativos, descripcion in analogias:
    resultados = analogia(positivos, negativos)
    print(f'\n{descripcion} =')
    for palabra, score in resultados[:3]:
        print(f'   → {palabra:<18} ({score:.3f})')

print()
print('Tu turno: explora tus propias analogías')
print("Ejemplo: analogia(['doctor', 'woman'], ['man'])")


## Experimento 7: Visualización con PCA

#### Reducción de dimensionalidad con PCA
Los vectores de GloVe tienen 100 dimensiones: imposible de visualizar directamente.
**PCA (Principal Component Analysis)** encuentra las 2 direcciones de mayor varianza
en el espacio y proyecta todos los vectores sobre esas 2 dimensiones.

Si los embeddings capturan semántica correctamente, palabras del mismo campo semántico
deberían quedar agrupadas en el espacio 2D resultante.

#### Interpretación del gráfico
- Las dos dimensiones (PC1 y PC2) no tienen un significado lingüístico directo.
- Lo que importa es la **posición relativa** de los puntos: palabras cercanas comparten contextos.
- La varianza explicada (%) indica cuánta información de las 100 dimensiones originales
  conserva la proyección. Un valor bajo (< 30%) indica que mucha información se perdió.


In [ ]:
# @title Experimento 7: Visualización de clusters semánticos con PCA
grupos = {
    'Financiero':   {'words': ['bank','credit','loan','rate','debt','investment',
                               'stock','market','finance','currency'], 'color':'#1B3A6B'},
    'Países LATAM': {'words': ['colombia','peru','mexico','chile','argentina',
                               'brazil','venezuela','ecuador','bolivia','uruguay'], 'color':'#16A34A'},
    'Mascotas':     {'words': ['cat','dog','fish','bird','rabbit',
                               'hamster','turtle','puppy','kitten','pet'],     'color':'#E8821A'},
    'Tecnología':   {'words': ['computer','software','algorithm','data','network',
                               'database','server','cloud','python','code'],   'color':'#7C3AED'},
}

palabras, vectores_lista, etiquetas, colores = [], [], [], []
for grupo, info in grupos.items():
    for word in info['words']:
        if word in modelo:
            palabras.append(word)
            vectores_lista.append(modelo[word])
            etiquetas.append(grupo)
            colores.append(info['color'])

vectores_np = np.array(vectores_lista)
pca = PCA(n_components=2, random_state=42)
v2d = pca.fit_transform(vectores_np)

fig, ax = plt.subplots(figsize=(13, 8))
ax.set_facecolor('#F8FAFF')
for i, (x, y) in enumerate(v2d):
    ax.scatter(x, y, color=colores[i], s=80, alpha=0.85, zorder=3)
    ax.annotate(palabras[i], (x, y), textcoords='offset points',
                xytext=(6, 4), fontsize=9, color=colores[i], fontweight='bold')

leyenda = [mpatches.Patch(color=info['color'], label=g) for g, info in grupos.items()]
ax.legend(handles=leyenda, loc='upper right', fontsize=10, framealpha=0.9)
ax.set_title('Espacio de Embeddings GloVe reducido a 2D con PCA', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza explicada)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza explicada)')
ax.grid(True, alpha=0.25, linestyle='--')
plt.tight_layout()
plt.show()

total = sum(pca.explained_variance_ratio_[:2]) * 100
print(f'Las 2 primeras componentes explican el {total:.1f}% de la varianza total.')


## Experimento 8 (Bonus): Auditoría de Sesgos de Género

#### El problema: sesgos heredados del corpus
Los word embeddings no aprenden solo semántica neutra: aprenden también los
**sesgos culturales y sociales** presentes en los textos de entrenamiento.

Bolukbasi et al. (2016) documentaron que Word2Vec entrenado en noticias de Google
exhibía sesgos de género sistemáticos:
```
doctor  - hombre + mujer ≈ enfermera
programador - hombre + mujer ≈ ama de casa
```

El modelo **no inventó** estos sesgos: los aprendió del texto escrito por humanos durante décadas.
Es un espejo de los patrones lingüísticos del corpus de entrenamiento.

#### ¿Por qué importa en producción?
Si usas embeddings sesgados en:
- **Sistemas de selección de CVs**: el modelo puede penalizar sistemáticamente a mujeres para roles técnicos
- **Sistemas de análisis de crédito**: puede reproducir discriminación histórica
- **Sistemas de moderación**: puede clasificar incorrectamente texto por región dialectal

#### ¿Qué hacer al detectar sesgo?
1. **Documentar** el sesgo antes de desplegar (Model Cards)
2. **Debiasing técnico**: proyectar embeddings fuera de la dirección de género (Bolukbasi et al.)
3. **Evaluar impacto real**: medir métricas separadas por grupo demográfico en la tarea downstream
4. **Curar el corpus**: balancear representación de género en los datos de entrenamiento


In [ ]:
# @title Experimento 8 (Bonus): Auditoría de sesgos de género
profesiones = ['doctor','nurse','engineer','teacher','programmer',
               'secretary','pilot','chef','lawyer','scientist','manager','assistant']

print(f'{"Profesión":<15} {"sim(man)":>9} {"sim(woman)":>10} {"Diferencia":>11} {"Orientación"}')
print('-' * 62)

datos_sesgo = []
for prof in profesiones:
    if prof in modelo:
        sim_h = modelo.similarity(prof, 'man')
        sim_m = modelo.similarity(prof, 'woman')
        diff  = sim_h - sim_m
        orient = 'hombre' if diff > 0.05 else ('mujer' if diff < -0.05 else 'neutro')
        print(f'{prof:<15} {sim_h:>9.3f} {sim_m:>10.3f} {diff:>11.3f} → {orient}')
        datos_sesgo.append((prof, sim_h, sim_m, diff))

fig, ax = plt.subplots(figsize=(10, 5))
profs  = [d[0] for d in datos_sesgo]
diffs  = [d[3] for d in datos_sesgo]
colors = ['#1B3A6B' if d > 0 else '#E8821A' for d in diffs]
ax.barh(profs, diffs, color=colors, alpha=0.8, edgecolor='white')
ax.axvline(0, color='black', linewidth=1.0)
ax.axvline(0.05,  color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(-0.05, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('sim(profesión, "man") − sim(profesión, "woman")\n(azul = sesgo hacia hombre, naranja = sesgo hacia mujer)')
ax.set_title('Sesgo de género en embeddings GloVe', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


### Reflexión Final

# === PAUSA: DISCUSIÓN FINAL ===

> **1. N-grams vs. Embeddings:**
> ¿En qué escenarios preferirías usar un modelo N-gram en lugar de embeddings GloVe?
> (Pista: piensa en latencia, recursos computacionales e interpretabilidad)
>
> **2. Embeddings estáticos:**
> La palabra "banco" tiene el mismo vector en "fui al banco a depositar" y
> "me senté en el banco del parque". ¿Qué consecuencia tiene esto para un sistema
> de análisis de sentimiento sobre noticias financieras?
>
> **3. Sesgos en producción:**
> Si tu empresa despliega un sistema de clasificación de CVs basado en GloVe,
> ¿qué auditoría harías antes de lanzarlo? ¿Qué métricas reportarías?
>
> **4. Gancho para la Semana 2:**
> BERT resuelve el problema de los embeddings estáticos con atención contextual.
> ¿Cómo crees que lo hace?

---

## Conceptos Clave del Lab

| Concepto | Definición | Limitación principal |
|---|---|---|
| **Preprocesamiento** | Transformar texto crudo en forma normalizada | No hay pipeline universal |
| **Tokenización por palabras** | Dividir por espacios/puntuación | Problema OOV |
| **BPE/WordPiece** | Subpalabras frecuentes, estándar en LLMs | 1 token no es 1 palabra |
| **Modelo N-gram** | Probabilidades de secuencias por conteo | Solo contexto local de N-1 palabras |
| **Perplexidad** | Métrica de calidad de modelos de lenguaje | Comparar solo con mismo vocabulario |
| **Word Embedding** | Vector denso que captura semántica | Embeddings estáticos: mismo vector por tipo |
| **Similitud coseno** | Distancia angular entre vectores | No captura relaciones asimétricas |
| **Analogía vectorial** | Aritmética en el espacio semántico | Funciona mejor con relaciones regulares |
| **Sesgo** | Prejuicio heredado del corpus | Difícil de eliminar completamente |
